Flight Delay Prediction (ML + DL)

In [ ]:
import pandas as pd
import random
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch

In [ ]:
airlines = ["Indigo", "Air India", "SpiceJet", "Vistara"]
airports = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata"]

data = []

for _ in range(500):
    airline = random.choice(airlines)
    source = random.choice(airports)
    destination = random.choice(airports)

    while source == destination:
        destination = random.choice(airports)

    distance = random.randint(300, 2500)
    departure_hour = random.randint(0, 23)
    weather = random.choice(["Clear", "Rain", "Fog"])

    if weather in ["Rain", "Fog"] or departure_hour > 20:
        delay = 1
    else:
        delay = 0

    data.append([airline, source, destination, distance, departure_hour, weather, delay])

df = pd.DataFrame(data, columns=[
    "airline", "source", "destination", "distance",
    "departure_hour", "weather", "delay"
])

df.to_csv("flights.csv", index=False)
print("Dataset created ")


In [ ]:
df = pd.read_csv("flights.csv")

X = df.drop("delay", axis=1)
y = df["delay"]

encoders = {}
for col in ["airline", "source", "destination", "weather"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [ ]:
model_ml = RandomForestClassifier()
model_ml.fit(X_train.numpy(), y_train.numpy())

accuracy_ml = model_ml.score(X_test.numpy(), y_test.numpy())
print("ML Accuracy:", accuracy_ml)

In [ ]:
class FlightModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

model_dl = FlightModel(X_train.shape[1])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_dl.parameters(), lr=0.001)

for epoch in range(50):
    outputs = model_dl(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

In [ ]:
with torch.no_grad():
    outputs = model_dl(X_test)
    _, predicted = torch.max(outputs, 1)

    accuracy_dl = (predicted == y_test).sum().item() / len(y_test)

print("DL Accuracy:", accuracy_dl)

In [ ]:
def predict_flight(airline, source, destination, distance, hour, weather):
    data = pd.DataFrame([[airline, source, destination, distance, hour, weather]],
                        columns=X.columns)

    for col in ["airline", "source", "destination", "weather"]:
        data[col] = encoders[col].transform(data[col])

    data = scaler.transform(data)
    tensor = torch.tensor(data, dtype=torch.float32)

    with torch.no_grad():
        output = model_dl(tensor)
        _, pred = torch.max(output, 1)

    return "Delay" if pred.item() == 1 else "On Time"

In [ ]:
print(predict_flight("Indigo", "Delhi", "Mumbai", 1400, 22, "Fog"))